In [113]:
import torch
#functions are made like this. functions don't store its own gradients inside function classes.
#That's because functions can be used multiple times, and we don't want it to be that way.
class MySquare(torch.autograd.Function):
    @staticmethod
    def forward(ctx, X):
        ctx.save_for_backward(X)
        return X * X
    @staticmethod
    def backward(ctx, grad_output):
        (X, ) = ctx.saved_tensors
        return  grad_output * 2 * X #this doesn't count in autograd graph


In [117]:
x = torch.tensor(3., requires_grad=True)
y = MySquare.apply(x)#
z = MySquare.apply(y)

In [115]:
X = torch.tensor(1.2, requires_grad=True)
for i in range(5):
    X = MySquare.apply(X)
X

tensor(341.8219, grad_fn=<MySquareBackward>)

In [118]:
x.grad_fn

In [ ]:
X_grad_fn = X.grad_fn
for i in range(5):
    print(X_grad_fn.next_functions)
    X_grad_fn = X_grad_fn.next_functions[0][0]


((<torch.autograd.function.MySquareBackward object at 0x11fade340>, 0),)
((<torch.autograd.function.MySquareBackward object at 0x11fade040>, 0),)
((<torch.autograd.function.MySquareBackward object at 0x11fade240>, 0),)
((<torch.autograd.function.MySquareBackward object at 0x11faddb40>, 0),)
((<AccumulateGrad object at 0x11fa85480>, 0),)


In [137]:
X_grad_fn.__dir__()

['__new__',
 '__call__',
 '_register_hook_dict',
 'register_hook',
 'register_prehook',
 'name',
 '_sequence_nr',
 '_set_sequence_nr',
 'next_functions',
 'requires_grad',
 'metadata',
 '_input_metadata',
 'variable',
 '__doc__',
 '__repr__',
 '__hash__',
 '__str__',
 '__getattribute__',
 '__setattr__',
 '__delattr__',
 '__lt__',
 '__le__',
 '__eq__',
 '__ne__',
 '__gt__',
 '__ge__',
 '__init__',
 '__reduce_ex__',
 '__reduce__',
 '__subclasshook__',
 '__init_subclass__',
 '__format__',
 '__sizeof__',
 '__dir__',
 '__class__']

In [107]:
z.grad_fn, type(y.grad_fn)

(<torch.autograd.function.MySquareBackward at 0x11fade140>,
 torch.autograd.function.MySquareBackward)

In [108]:
z.grad_fn.next_functions[0][0], y.grad_fn

(<torch.autograd.function.MySquareBackward at 0x11fadde40>,
 <torch.autograd.function.MySquareBackward at 0x11fadde40>)

In [109]:
class MyMult(torch.autograd.Function):
    @staticmethod
    def forward(ctx, X, Y):
        ctx.save_for_backward(X, Y)
        return X * Y
    @staticmethod
    def backward(ctx, grad_output):
        (X, Y,) = ctx.saved_tensors
        grad_X = grad_output * Y
        grad_Y = grad_output * X
        print('dd')
        return  grad_X, grad_Y

In [110]:
X1 = torch.tensor(1., requires_grad=True)
Y1 = MySquare.apply(X1)
X2 = torch.tensor(2., requires_grad=True)
Y2 = MySquare.apply(X2)
Z = MyMult.apply(Y1, Y2)

In [129]:
Z.grad_fn._forward_cls

__main__.MyMult

In [120]:
Z.grad_fn.__dir__()

['_forward_cls',
 '__module__',
 '__doc__',
 '_autograd_function_id',
 '_compiled_autograd_should_lift',
 'apply',
 'apply_jvp',
 '_compiled_autograd_key',
 '__dict__',
 '__weakref__',
 '__new__',
 'name',
 '_sequence_nr',
 '_set_sequence_nr',
 'maybe_clear_saved_tensors',
 '_register_hook_dict',
 'register_hook',
 'register_prehook',
 '_is_compiled_autograd_tracing',
 '_get_compiled_autograd_symints',
 'saved_tensors',
 'saved_variables',
 '_raw_saved_tensors',
 'next_functions',
 'to_save',
 'non_differentiable',
 'dirty_tensors',
 'saved_for_forward',
 'needs_input_grad',
 'requires_grad',
 'metadata',
 '_input_metadata',
 'materialize_grads',
 '_materialize_non_diff_grads',
 '_compiled_autograd_backward_state',
 '__repr__',
 '__hash__',
 '__str__',
 '__getattribute__',
 '__setattr__',
 '__delattr__',
 '__lt__',
 '__le__',
 '__eq__',
 '__ne__',
 '__gt__',
 '__ge__',
 '__init__',
 '__reduce_ex__',
 '__reduce__',
 '__subclasshook__',
 '__init_subclass__',
 '__format__',
 '__sizeof__',

In [166]:
X1 = torch.tensor(1., requires_grad=True)
Y1 = MySquare.apply(X1)
X2 = torch.tensor(2., requires_grad=True)
Y2 = MySquare.apply(X2)
Y2 = Y2.detach()
Y2_2 = MySquare.apply(Y2).requires_grad_(True)
Z = MyMult.apply(Y1, Y2_2)



In [167]:
Y2_2 = Y2.detach()

In [168]:
Z.grad_fn.next_functions

((<torch.autograd.function.MySquareBackward at 0x11fb03a40>, 0),
 (<AccumulateGrad at 0x11fa870a0>, 0))

In [198]:
X, Y

(tensor(341.8219, grad_fn=<MySquareBackward>),
 tensor(2., grad_fn=<MyMultBackward>))

In [214]:
Z = (X + Y)
Z.backward()

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

In [ ]:

X.requires_grad_(True)
Z = (X + Y)
Z.backward()
X.grad = None
Y.grad = None
Z.grad = None
Z = (X + Y)
Z.backward()

In [207]:
print(X.grad)

None


/var/folders/w7/73t892td0lv16vj9rdl7098w0000gn/T/ipykernel_8519/3058899600.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1729647065806/work/build/aten/src/ATen/core/TensorBody.h:494.)
  print(X.grad)
